# 训练期现金转SOXL局部动作的30%权重合法性修复审计

只读原生Qlib父账本、原封存标签和已关闭因子来源；保留原Recorder，不重算非法动作收益，只在合法子集核对旧结论。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
root=Path.cwd();card=root/'research/stage1_soxl_local_action_max_weight_legality_audit_20261024'
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());s=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_market_GETs','new_model_fits','new_native_accounts','new_label_passes','new_bootstraps','new_Recorders'))
for name,key in [('reference_card','reference_summary_sha256'),('SOXL_card','SOXL_summary_sha256'),('SOXX_card','SOXX_summary_sha256'),('QQQ_card','QQQ_summary_sha256')]:assert sha(root/s[name]/'summary.json')==s[key]
assert sha(root/s['provider']/'qlib_store_manifest.json')==s['provider_manifest_sha256']
ref=root/s['reference_card'];a=root/s['SOXL_card'];b=root/s['SOXX_card'];q=root/s['QQQ_card']
for source,exp,rid,art,key in [(ref,'reference_experiment_id','reference_recorder_id','paths/reference_0','reference_path_sha256'),(ref,'reference_experiment_id','reference_recorder_id','inputs/action_plan','reference_action_plan_sha256'),(a,'SOXL_experiment_id','SOXL_recorder_id','analysis/labels','SOXL_labels_sha256'),(b,'SOXX_experiment_id','SOXX_recorder_id','analysis/labels','SOXX_labels_sha256'),(q,'QQQ_experiment_id','QQQ_recorder_id','analysis/source_frame','QQQ_frame_sha256')]:
 assert sha(source/'qlib_recorder'/s[exp]/s[rid]/'artifacts'/art)==s[key],key
sr=json.loads((ref/'summary.json').read_text());sa=json.loads((a/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text());sq=json.loads((q/'summary.json').read_text())
assert sr['nas_readback']['verified'] and sa['nas_readback']['verified'] and sa['nas_readback']['sha256']==s['SOXL_NAS_sha256']
assert sb['nas_readback']['verified'] and sq['nas_readback']['verified']
qlib.init(provider_uri=str((root/s['provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((ref/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['reference_experiment_id'],recorder_id=s['reference_recorder_id']);bundle=rec.load_object('paths/reference_0');action=rec.load_object('inputs/action_plan')
with R.uri_context(str((a/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['SOXL_experiment_id'],recorder_id=s['SOXL_recorder_id']);lab=rec.load_object('analysis/labels')
with R.uri_context(str((b/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['SOXX_experiment_id'],recorder_id=s['SOXX_recorder_id']);old=rec.load_object('analysis/labels')
with R.uri_context(str((q/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['QQQ_experiment_id'],recorder_id=s['QQQ_recorder_id']);qqq=rec.load_object('analysis/source_frame')
path=bundle['replay'].reset_index(drop=True);targets=bundle['targets'].reset_index(drop=True)
assert len(path)==len(targets)==len(lab)==len(old)==1061
trade=pd.DatetimeIndex(lab.trade_date);signal=pd.DatetimeIndex(lab.signal_date)
assert trade.equals(pd.DatetimeIndex(path.trade_date)) and trade.equals(pd.DatetimeIndex(targets.trade_date)) and trade.equals(pd.DatetimeIndex(old.trade_date))
assert trade.equals(pd.DatetimeIndex(D.calendar(start_time='2018-10-15',end_time='2022-12-30',freq='day')))
assert signal.equals(pd.DatetimeIndex(path.signal_date)) and signal.isin(qqq.index).all()
cap=float(action['recovery_spec']['maximum_soxl_weight']);assert cap==.30
base=targets.soxl_weight.to_numpy(float);delta=lab.delta_SOXL_weight.to_numpy(float);candidate=base+delta
assert np.isfinite(base).all() and np.isfinite(delta).all() and (base<=cap+1e-10).all()
old_eligible=lab.eligible.to_numpy(bool);old_mature=lab.soxl_local_fee_net_advantage.notna().to_numpy(bool)
invalid=candidate>cap+1e-10;legal=~invalid
assert old_eligible.sum()==689 and old_mature.sum()==688 and np.all(invalid<=old_eligible)
score_drawdown=-path.portfolio_drawdown.to_numpy(float)
score_qqq=-qqq.loc[signal,'downside_beta_gap_60'].to_numpy(float)
assert np.isfinite(score_drawdown).all() and np.isfinite(score_qqq).all()
x=pd.DataFrame({'signal_date':signal,'trade_date':trade,'year':trade.year,'quarter':trade.quarter,
                'risk_state':path.risk_state.to_numpy(int),'cap_binding':lab.cap_binding.to_numpy(bool),
                'eligible':old_eligible,'mature':old_mature,'invalid':invalid,'legal':legal,
                'base_soxl':base,'delta_soxl':delta,'candidate_soxl':candidate,
                'label':lab.soxl_local_fee_net_advantage.to_numpy(float),
                'old_SOXX_label':old.joint_local_fee_net_advantage.to_numpy(float),
                'drawdown_score':score_drawdown,'QQQ_score':score_qqq})
legal_mature=x[x.legal&x.mature].copy()
def local_stats(g):
 k=g[g.mature]
 return {'sessions':len(g),'old_eligible':int(g.eligible.sum()),'old_mature':int(g.mature.sum()),'invalid_eligible':int((g.invalid&g.eligible).sum()),
         'invalid_mature':int((g.invalid&g.mature).sum()),'legal_mature':int((g.legal&g.mature).sum()),
         'legal_mean':float(g.loc[g.legal&g.mature,'label'].mean()) if (g.legal&g.mature).any() else None,
         'legal_positive':int((g.loc[g.legal&g.mature,'label']>0).sum())}
annual=[{'year':int(y),**local_stats(g)} for y,g in x.groupby('year',sort=True)]
quarter=[{'year':int(y),'quarter':int(qt),**local_stats(g)} for (y,qt),g in x.groupby(['year','quarter'],sort=True)]
by_binding={str(v).lower():local_stats(g) for v,g in x.groupby('cap_binding')}
assert len(annual)==5 and len(quarter)==17

def rank(g,name):
 k=g[g.legal&g.mature];z=k[k.label.abs()>1e-12];y=(z.label>0).to_numpy(int)
 auc=float(roc_auc_score(y,z[name].to_numpy(float))) if len(set(y))==2 else None
 sp=float(spearmanr(k[name].to_numpy(float),k.label.to_numpy(float)).statistic) if len(k)>2 and k[name].nunique()>1 and k.label.nunique()>1 else None
 return {'mature':len(k),'positive':int((k.label>1e-12).sum()),'negative':int((k.label<-1e-12).sum()),'zero':int((k.label.abs()<=1e-12).sum()),'auc':auc,'spearman':sp}
corrected={}
for name in ('drawdown_score','QQQ_score'):
 full=rank(x,name);years=[{'year':y,**rank(x[x.year==y],name)} for y in (2020,2021,2022)]
 eras=[{'era':'2020_2021',**rank(x[x.year.isin([2020,2021])&x.cap_binding],name)},{'era':'2022',**rank(x[(x.year==2022)&x.cap_binding],name)}]
 gates={'each_2020_2022_auc_above_half':all(z['auc'] is not None and z['auc']>.5 for z in years),
        'each_2020_2022_spearman_positive':all(z['spearman'] is not None and z['spearman']>0 for z in years),
        'binding_era_spearman_positive':all(z['spearman'] is not None and z['spearman']>0 for z in eras)}
 corrected[name]={'full':full,'complete_years':years,'binding_eras':eras,'original_gates_on_legal_subset':gates,'necessary_gate_pass':all(gates.values())}
comp=legal_mature[legal_mature.old_SOXX_label.notna()]
compare={'matched_legal_mature':len(comp),'SOXL_mean':float(comp.label.mean()),'SOXX_mean':float(comp.old_SOXX_label.mean()),
         'SOXL_minus_SOXX_mean':float((comp.label-comp.old_SOXX_label).mean()),'correlation':float(comp[['label','old_SOXX_label']].corr().iloc[0,1])}
matched=[]
for state in range(4):
 a0=legal_mature[(legal_mature.risk_state==state)&legal_mature.year.isin([2020,2021])&legal_mature.cap_binding]
 b0=legal_mature[(legal_mature.risk_state==state)&(legal_mature.year==2022)&legal_mature.cap_binding]
 matched.append({'risk_state':state,'earlier':len(a0),'later':len(b0),'earlier_mean':float(a0.label.mean()) if len(a0) else None,'later_mean':float(b0.label.mean()) if len(b0) else None,'matched':bool(len(a0)>=10 and len(b0)>=10)})
matched_ok=[v for v in matched if v['matched']];matched_2022=sum(v['later'] for v in matched_ok)
result={'status':'audit_invalid_SOXL_max_weight_nonbinding_labels_preserved_legal_binding_only','plan_sha256':sha(plan_path),
        'source_recorder_ids':{'reference':s['reference_recorder_id'],'SOXL':s['SOXL_recorder_id'],'SOXX':s['SOXX_recorder_id'],'QQQ':s['QQQ_recorder_id']},
        'maximum_soxl_weight':cap,'full_sessions':len(x),'old_eligible':int(old_eligible.sum()),'old_mature':int(old_mature.sum()),
        'invalid_eligible':int((invalid&old_eligible).sum()),'invalid_mature':int((invalid&old_mature).sum()),
        'legal_eligible':int((legal&old_eligible).sum()),'legal_mature':len(legal_mature),
        'max_old_candidate_soxl':float(candidate.max()),'annual':annual,'quarter':quarter,'by_binding':by_binding,
        'legal_local_mean':float(legal_mature.label.mean()),'legal_SOXX_comparison':compare,'legal_risk_state_overlap':{'cells':matched,'matched_cells':len(matched_ok),'matched_2022':matched_2022,'full_2022_legal':int((legal_mature.year==2022).sum())},
        'corrected_necessary_ranks':corrected,'new_market_GETs':0,'new_model_fits':0,'new_native_accounts':0,'new_label_passes':0,'new_bootstraps':0,'new_Recorders':0,
        'SOXX_goal_achieved':False,'limits':['Original fee-net labels on illegal candidate targets remain in frozen Recorder but are not legal action evidence','Masking invalid labels is an audit of the old candidate, not a new capped target or replay','Binding-only local labels are training history and not a full exact-f696 own-state strategy or independent forward validation']}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'invalid':result['invalid_mature'],'legal':result['legal_mature'],'by_year':[(r['year'],r['invalid_mature'],r['legal_mature']) for r in annual],
       'corrected_gates':{k:v['original_gates_on_legal_subset'] for k,v in corrected.items()},'legal_SOXX':compare})


{'invalid': 148, 'legal': 540, 'by_year': [(2018, 1, 18), (2019, 99, 39), (2020, 20, 182), (2021, 26, 56), (2022, 2, 245)], 'corrected_gates': {'drawdown_score': {'each_2020_2022_auc_above_half': False, 'each_2020_2022_spearman_positive': False, 'binding_era_spearman_positive': False}, 'QQQ_score': {'each_2020_2022_auc_above_half': False, 'each_2020_2022_spearman_positive': False, 'binding_era_spearman_positive': False}}, 'legal_SOXX': {'matched_legal_mature': 540, 'SOXL_mean': 5.147577602822252e-05, 'SOXX_mean': 9.534886586010744e-05, 'SOXL_minus_SOXX_mean': -4.387308983188489e-05, 'correlation': 0.9950319510903469}}


## 中文结论

# SOXL局部动作30%上限漏检：148个旧标签无效

原架构2018–2022训练账本明确规定 `maximum_soxl_weight=0.30`。在同一1061日Qlib原生来源上逐日复算旧候选 `原投影SOXL权重+原增配权重`，原投影本身均合法，但旧候选 **689个资格日中148个越过30%**，最大到 **31.1338%**；148个都已成熟，且**恰为全部旧“未受beta上限约束”动作**。原局部标签计算检查了现金和beta，没有检查SOXL最高权重。这是实质性的合法性错误：旧688日全体均值、148日未受限价值及基于全体的因子排序不能当作合法动作证据。原Recorder原字节、费用计算与NAS回读保留，不把非法分支事后截到30%冒充旧含费标签。

按原固定动作只保留没有越界的 **540个成熟受限动作**（541个合法资格日，末日标签未知）。2018–2022各年合法成熟数18/39/182/56/245；合法分支平均局部净优势 **+0.00005148**，2022 **−0.00016302**。与旧SOXX同日同beta分支在540日匹配的SOXL/SOXX均值为 **+0.00005148/+0.00009535**，SOXL减SOXX **−0.00004387**，相关 **0.99503**。2022的245个合法标签全处在此前可比的四个受限风险状态格子里，四格跨年差异仍在；先前“245/247覆盖99.2%”的分母含2个非法动作，应更正为**245/245**。

在合法540日上按原方向和原公式复算必要排序：负回撤全段AUC **0.52190**，2020/2021/2022为 **0.41894/0.54809/0.45519**；负SOXX/QQQ下跌beta缺口全段AUC **0.47844**，三年为 **0.52185/0.50461/0.47050**。两条原固定必要门在合法子集上仍全败。先前688日的全段和年度AUC只能作为错误分析历史，不能与这些合法值混用。审计0新行情、拟合、动作重放、标签、资金账户或Recorder；来源SHA、NAS回读和Qlib日历均通过。

决定 `invalid_148_nonbinding_SOXL_local_labels_legal_binding_540_reaudit`。训练期现金转SOXL固定版本仍关闭；若未来研究未受限层，必须另冻**受30%上限约束的新动作**并重新计算费用与自身账户，不能复用旧148个标签。当前没有精确f696新资金账户、SOXX净超额、2/5bp或独立前向证据。
